# Lab 03 — Competitive learning and SOM

**Real dataset:** [Beothuk/uci-har-federated](https://huggingface.co/datasets/Beothuk/uci-har-federated). We use a small feature subset from human-activity recognition and ignore labels during SOM training. The labels are reserved for post-hoc interpretability.

Goal: build a simple winner-take-all prototype learner, extend it to a 2-D SOM, and quantify topology/cluster organization.

## The math behind the experiment

For an input $x$, competitive learning chooses the prototype with the smallest distance:

$$
k^*=\arg\min_k\|x-w_k\|_2.
$$

The winning prototype then moves toward the example:

$$
w_{k^*}\leftarrow w_{k^*}+\eta(x-w_{k^*}).
$$

A Self-Organizing Map extends this idea: nearby map units can also move, usually with a distance-dependent neighborhood weight $h(k,k^*)$.

The important idea is simple: **the model is not given class labels; it organizes examples by similarity.**

In [ ]:
!pip -q install datasets scikit-learn pandas numpy matplotlib
from datasets import load_dataset
import numpy as np,pandas as pd
from sklearn.preprocessing import StandardScaler
raw=load_dataset("Beothuk/uci-har-federated",split="train[:1500]")
print(raw.features)
# Robustly select numeric feature columns from the returned records.
rows=raw.to_pandas(); num=rows.select_dtypes(include=np.number)
X=StandardScaler().fit_transform(num.iloc[:,:min(12,num.shape[1])].fillna(0).values)
print("shape",X.shape,"numeric columns",list(num.columns[:min(12,num.shape[1])]))

## Step 1 — Predict

A winner-take-all learner should place prototypes near dense regions. A SOM should additionally make neighboring map cells represent nearby input regions. It is **not** expected to discover labels directly because labels are not used during training.

In [ ]:
def train_prototypes(X,k=8,epochs=8,lr=.2,seed=0):
 rng=np.random.default_rng(seed); W=X[rng.choice(len(X),k,replace=False)].copy()
 for _ in range(epochs):
  for x in X[rng.permutation(len(X))]:
   j=np.argmin(((W-x)**2).sum(1)); W[j]+=lr*(x-W[j])
 return W
W=train_prototypes(X); print("prototype shape",W.shape)
# Quantization error: average distance to winning prototype.
qe=np.mean([np.min(((W-x)**2).sum(1))**.5 for x in X]); print("quantization_error",qe)

In [ ]:
def train_som(X,shape=(4,4),epochs=10,lr=.25,sigma=2.,seed=0):
 rng=np.random.default_rng(seed); n=shape[0]*shape[1]; W=X[rng.choice(len(X),n,replace=False)].copy(); coords=np.array([(i,j) for i in range(shape[0]) for j in range(shape[1])])
 for e in range(epochs):
  a=lr*(1-e/epochs); s=max(.5,sigma*(1-e/epochs))
  for x in X[rng.permutation(len(X))]:
   d=((W-x)**2).sum(1); win=np.argmin(d); grid=((coords-coords[win])**2).sum(1); h=np.exp(-grid/(2*s*s)); W += a*h[:,None]*(x-W)
 return W
S=train_som(X)
print("SOM quantization error",np.mean([np.min(((S-x)**2).sum(1))**.5 for x in X]))

## Step 2 — Intervention

Change only the SOM neighborhood width. A very broad neighborhood should encourage smooth maps but can reduce local specialization; a narrow neighborhood can fit local structure while losing global ordering.

In [ ]:
rows=[]
for sig in [.5,1.,2.,4.]:
 S=train_som(X,sigma=sig,seed=0); qe=np.mean([np.min(((S-x)**2).sum(1))**.5 for x in X]); rows.append([sig,qe])
results=pd.DataFrame(rows,columns=["initial_sigma","quantization_error"]).sort_values("quantization_error"); results

## Conclusion / answer key

Expected: prototype learning reduces quantization error compared with random prototypes; SOM neighborhood choice changes the trade-off between local fit and map smoothness. The correct conclusion is about **representation geometry**, not classification accuracy.

### Research extension
Compare SOM map sizes at a fixed prototype count and evaluate topology preservation with a neighborhood-preservation metric.